# Laboratory, warm start and the first migration

This notebook shows:
- the complete catalogue of the source archive and its statuses;
- source aliases and deferred variants;
- seed spins → q → product state;
- two different warm-start mixers;
- a full run with a budget of 544;
- checkpoint/resume;
- a real classical source;
- a small controlled series H.

The main example uses only NumPy and `provided_spins`, so it runs in the CPU
profile without CVXPY or CUDA.

In [1]:
from pathlib import Path
import copy
import json
import os
import tempfile

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

from QA_max_cut_lib import GraphFactory, RunControl, WeightedGraph, restore, tools
from QA_max_cut_lib.laboratory import WarmStartQAOACPU, catalog_summary, get_file, get_variant, list_variants
from QA_max_cut_lib.solvers.warm_start import product_state, state_at_warm_angles
from QA_max_cut_lib.solvers.qaoa_state import cut_diagonal

print(catalog_summary())
for card in list_variants():
    print(card["variant_id"], card["runnable"], card["status"]["implementation"])
assert len(list_variants()) >= 6


{'archive_sha256': 'b4c684a5d4da469adc25bd99a405e402892d1a2524465e4c97ab6229c71a4aed', 'variants': 6, 'source_members': 89, 'parsed_members': 89, 'implemented_variant_ids': ['warm_start_product_qaoa_cpu_v1'], 'deferred_variant_ids': ['legacy_ls_rqaoa_fast_alias', 'warm_start_rqaoa', 'laboratory_ls_families', 'mutual_information_diagnostics', 'deferred_environment_and_generators']}
warm_start_product_qaoa_cpu_v1 True implemented
legacy_ls_rqaoa_fast_alias False not_implemented
warm_start_rqaoa False not_implemented
laboratory_ls_families False not_implemented
mutual_information_diagnostics False not_implemented
deferred_environment_and_generators False not_implemented


## 1. Inventory and aliases

All 89 Python members of the archive have a file record. `Diplom/Mk_5.py` and `Laboratory/LS_RQAOA_fast_5.py` share the same AST; this is a
provisional equivalence, not proof of an identical trajectory.

In [2]:
left = get_file("Diplom/Mk_5.py")
right = get_file("Laboratory/LS_RQAOA_fast_5.py")
print(left["path"], left["sha256"])
print(right["path"], right["sha256"])
print("same AST:", left["ast_sha256"] == right["ast_sha256"])
print("same bytes:", left["sha256"] == right["sha256"])
deferred = get_variant("warm_start_rqaoa")
print("WS_RQAOA:", deferred["status"])
assert left["ast_sha256"] == right["ast_sha256"]
assert left["sha256"] != right["sha256"]


Diplom/Mk_5.py 34a3dd3ba5346583456d281646a5272cad7506dae09bbb0bb0f90948b780fe71
Laboratory/LS_RQAOA_fast_5.py 91006f32df8dc04e705e834551a0219f37ae1134253d58816dae2ce3d69c2b5c
same AST: True
same bytes: False
WS_RQAOA: {'source_review': 'inventoried', 'scientific_specification': 'hypothesis', 'implementation': 'not_implemented', 'cpu_validation': 'not_run', 'gpu_validation': 'not_supported', 'resume_validation': 'not_supported'}


## 2. Spins, probabilities and product state A

Nodes 10,30, edge weight 2, seed `(+1,-1)`, epsilon=1/4. Expected probabilities:
`(3/16, 1/16, 9/16, 3/16)`, initial expected C=5/4.

In [3]:
graph = WeightedGraph([(10, 30, 2)], nodes=[10, 30])
q_raw = np.array([0.0, 1.0])
q = np.array([0.25, 0.75])
state = product_state(q)
probabilities = np.abs(state) ** 2
cuts = cut_diagonal(graph)
print("amplitudes:", state)
print("probabilities:", probabilities)
print("initial C:", float(cuts @ probabilities))
print("source cut:", graph.cut_value({10: 1, 30: -1}))
assert np.allclose(probabilities, [3/16, 1/16, 9/16, 3/16])
assert np.isclose(cuts @ probabilities, 1.25)


amplitudes: [0.4330127+0.j 0.25     +0.j 0.75     +0.j 0.4330127+0.j]
probabilities: [0.1875 0.0625 0.5625 0.1875]
initial C: 1.25
source cut: 2.0


## 3. Two warm-start mixers

At epsilon=1/2 `rounded_modified` must coincide with the ordinary X mixer at the
same beta, and `warm_aligned` at `-beta`. Both energies and full distributions
are compared here.

In [4]:
from QA_max_cut_lib import QAOACPU

params = {"layers": 1, "max_vertices": 8, "dtype": "complex128"}
base = {
    "type": "qaoa_cpu", "seed": 1, "parameters": params,
    "objective": {"type": "mean_cut"},
    "estimation": {"mode": "exact"},
    "optimizer": {"type": "adam", "parameters": {"max_steps": 0}},
    "initialization": {"type": "fixed_angles", "parameters": {"values": [0.4, 0.7]}},
    "readout": {"mode": "most_probable"}, "limits": {"shots_total": 0},
    "collect": {"probabilities": True}, "output": {"report": 0, "logs": 0},
}
warm = copy.deepcopy(base)
warm["type"] = "laboratory.warm_start_qaoa_cpu"
warm["warm_start"] = {
    "epsilon": 0.5,
    "source": {"type": "provided_spins", "parameters": {"entries": [{"node": 10, "spin": 1}, {"node": 30, "spin": -1}]}},
    "mixer": {"type": "rounded_modified"},
}
warm["parameters"] = params
warm_solver = WarmStartQAOACPU(warm)
warm_solver.solve(graph)
standard_solver = QAOACPU(base)
standard_solver.solve(graph)
print("max probability difference:", float(np.max(np.abs(warm_solver.probabilities - standard_solver.probabilities))))
assert np.allclose(warm_solver.probabilities, standard_solver.probabilities, atol=1e-12)


max probability difference: 0.0


## 4. Full provided-spin run

p=1, three ADAM iterations, 32 shots/evaluation, 64 readout: total budget
`3*5*32 + 64 = 544`. The source cut and the expected product cut are different
fields.

In [5]:
config = tools.load_json("configs/solvers/warm_start_qaoa_cpu_provided.json")
config = copy.deepcopy(config)
config["checkpoint"]["path"] = str(Path(tempfile.mkdtemp(prefix="qa_a11_catalog_")) / "run")
config["output"] = {"report": 0, "logs": 0, "color": "never"}
g4 = GraphFactory.from_recipe(tools.load_json("configs/graphs/warm_start_4.json"))
provided = WarmStartQAOACPU(config)
solution = provided.solve(g4)
print("solution:", solution)
print("cut:", provided.cut_value, "source_cut:", provided.source_cut)
print("initial expected cut:", provided.initial_expected_cut)
print("mixer:", provided.mixer_id)
print("resources:", provided.resources)
assert provided.resources["shots_used"] == 544
assert provided.source_cut is not None


solution: {-7: -1, 10: 1, 80: 1, 101: -1}
cut: 2.0 source_cut: -3.0
initial expected cut: -1.5
mixer: rounded_modified
resources: {'shots_used': 544, 'known_spent': 544, 'uncertain_upper_charge': 0, 'total_charged': 544, 'training_shots': 480, 'logical_training_shots': 480, 'readout_shots': 64, 'readout_reserved': 0, 'shots_available': 0, 'objective_evaluations': 15, 'optimizer_steps': 3, 'num_stages': 4, 'num_treads': 1, 'checkpoint_writes': 5, 'source_cost': None, 'warm_state_build': 0}


## 5. Pause → restore → resume

Pause after one completed iteration: 160 training shots and 64 held. Continuation
in a new object receives the same canonical run and the full budget.

In [6]:
pause_config = copy.deepcopy(config)
pause_config["checkpoint"] = {
    "enabled": True,
    "path": str(Path(tempfile.mkdtemp(prefix="qa_a11_pause_")) / "run"),
    "every": {"optimizer_iterations": 1},
}
paused = WarmStartQAOACPU(pause_config)
paused.solve(g4, control=RunControl(pause_after={"optimizer_iterations": 1}))
print("paused:", paused.status, paused.resources)
path = paused.last_checkpoint.path
continued = restore(path)
final = continued.resume()
print("final:", final, continued.resources)
print("same angles:", np.allclose(continued.optimal_angles, provided.optimal_angles))
assert paused.status == "paused"
assert continued.status == "completed"
assert continued.resources["shots_used"] == 544


paused: paused {'shots_used': 160, 'known_spent': 160, 'uncertain_upper_charge': 0, 'total_charged': 160, 'training_shots': 160, 'logical_training_shots': 160, 'readout_shots': 0, 'readout_reserved': 64, 'shots_available': 320, 'objective_evaluations': 5, 'optimizer_steps': 1, 'num_stages': 2, 'num_treads': 1, 'checkpoint_writes': 1, 'source_cost': None, 'warm_state_build': 0}
final: {-7: -1, 10: 1, 80: 1, 101: -1} {'shots_used': 544, 'known_spent': 544, 'uncertain_upper_charge': 0, 'total_charged': 544, 'training_shots': 480, 'logical_training_shots': 480, 'readout_shots': 64, 'readout_reserved': 0, 'shots_available': 0, 'objective_evaluations': 15, 'optimizer_steps': 3, 'num_stages': 4, 'num_treads': 1, 'checkpoint_writes': 5, 'source_cost': None, 'warm_state_build': 0}
same angles: True


## 6. A real classical source

GW and SA run lazily only when the source is selected. Their reads/hyperplanes
and time remain a separate `source_cost`; quantum shots are separate.

In [7]:
classical_config = tools.load_json("configs/solvers/warm_start_qaoa_cpu_gw.json")
classical_config = copy.deepcopy(classical_config)
classical_config["output"] = {"report": 0, "logs": 0, "color": "never"}
gw = WarmStartQAOACPU(classical_config)
gw.solve(g4)
print("source:", gw.source_type, "source_cut:", gw.source_cut)
print("source_cost:", gw.source_cost)
print("quantum_shots:", gw.resources["shots_used"])
assert gw.source_cost is not None
assert gw.resources["shots_used"] == 224


source: classical_solver source_cut: 2.0
source_cost: {'shots_used': 0, 'sdp_calls': 1, 'hyperplanes': 8, 'num_treads': 1, 'solve_seconds': 0.01883020019158721, 'total_seconds': 1.844138499815017}
quantum_shots: 224


## 7. Small series H

3 graphs × 2 algorithm seeds × 7 configurations (base + 2 epsilon × 3 mixer) =
42 runs. This checks the budget and an honest comparison, not a proof that warm
start is superior.

In [8]:
spec = tools.load_json("experiments/benchmark_warm_start.json")
spec.pop("save", None)
series = tools.ExperimentRunner(spec).run()
print(series.summary)
print("planned/valid:", series.analysis["planned"], series.analysis["valid"])
print("quantum shots per run: 544")
print("declared series shots:", 42 * 544)
assert series.analysis["planned"] == 42
assert series.analysis["valid"] == 42
assert series.analysis["failed"] == 0


planned=42 valid=42 failed=0 skipped=0 success(valid)=0.8333333333333334 wall=0.248s
planned/valid: 42 42
quantum shots per run: 544
declared series shots: 22848


## 8. Source and final distribution

The labels clearly separate the source cut, the initial expected cut and the final
cut. The plot is built from the real arrays of this run; the classical source is
not called again.

In [9]:
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.36
index = np.arange(len(provided.probabilities))
ax.bar(index - width / 2, np.abs(product_state(provided.warm_start_data["q_clipped"])) ** 2, width, label="initial product")
ax.bar(index + width / 2, provided.probabilities, width, label="final QAOA")
ax.set_xlabel("basis index (little-endian node order)")
ax.set_ylabel("probability")
ax.set_title("Warm start: source cut=%.1f, initial E[C]=%.3f, final C=%.3f" % (provided.source_cut, provided.initial_expected_cut, provided.cut_value))
ax.set_xticks(index)
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
figure_path = Path("docs/notebook_previews/12_laboratory_catalog_and_warm_start.png")
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=140, facecolor="white")
plt.close(fig)
print("saved:", figure_path)


saved: docs\notebook_previews\12_laboratory_catalog_and_warm_start.png


## 9. Comparison H

The table below is solver-only quality and quantum shots. `source_cost` for GW/SA
is not attributed to configurations without a classical source.

In [10]:
rows = []
for name, data in series.analysis["by_solver"].items():
    rows.append((name, data["cut"]["mean"], data["success"], data["valid"]))
for row in sorted(rows):
    print("%-24s mean_cut=%.3f success=%.3f n=%d" % row)
assert len(rows) == 7


base                     mean_cut=3.333 success=0.667 n=6
warm_e025_aligned        mean_cut=3.667 success=1.000 n=6
warm_e025_modified       mean_cut=3.667 success=1.000 n=6
warm_e025_x              mean_cut=3.500 success=0.833 n=6
warm_e050_aligned        mean_cut=3.667 success=1.000 n=6
warm_e050_modified       mean_cut=3.333 success=0.667 n=6
warm_e050_x              mean_cut=3.333 success=0.667 n=6


## Summary

- The catalogue covers 89 source files; deferred cards are visible.
- Product state A reproduces `(3/16,1/16,9/16,3/16)` and initial C=5/4.
- `rounded_modified` and `warm_aligned` have different signs of beta.
- A full provided run uses 544 shots.
- Pause/resume continues the canonical run.
- The GW source has a separate preprocessing cost.
- H is 42 real runs, not a claim of warm-start superiority.